# GUI Application

> Tkinter-based control panel for GzKeyboard

In [ ]:
#| default_exp gui

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import tkinter as tk
from tkinter import ttk
import sys

from gzkeyboard.system import SystemInputHandler, InputMode, KeyboardConfig, start_keyboard, stop_keyboard

## GUI Application

A small control panel window. The keyboard hook runs in the background while
tkinter's `mainloop()` keeps the app alive — no extra threads needed.

In [ ]:
#| export
class GzKeyboardApp:
    """Tkinter control panel for GzKeyboard.

    Provides Start/Stop, Pause, and mode selection controls.
    The keyboard hook runs in the background; tkinter mainloop keeps the app alive.
    """

    BG         = '#2b2b2b'
    FG         = '#ffffff'
    BTN_BG     = '#3c3f41'
    BTN_ACTIVE = '#4c5052'
    GREEN      = '#4caf50'
    YELLOW     = '#ffc107'
    RED        = '#f44336'
    ACCENT     = '#4fc3f7'

    MODES = [
        ('Tigrinya', 'tigrinya'),
        ('Amharic',  'amharic'),
        ('Latin',    'latin'),
    ]

    def __init__(self, root: tk.Tk):
        self.root = root
        self.handler: SystemInputHandler = None
        self.paused = False
        self._build_ui()
        self._update_ui()

    # --- UI construction ---

    def _build_ui(self):
        self.root.title('GzKeyboard')
        self.root.resizable(False, False)
        self.root.configure(bg=self.BG)
        self.root.protocol('WM_DELETE_WINDOW', self._on_close)

        pad = dict(padx=12, pady=6)

        # Status bar
        self.status_var = tk.StringVar(value='Stopped')
        self.status_label = tk.Label(
            self.root, textvariable=self.status_var,
            bg=self.BG, fg=self.FG,
            font=('Segoe UI', 11, 'bold'), anchor='w'
        )
        self.status_label.grid(row=0, column=0, columnspan=3, sticky='we', padx=12, pady=(12, 4))

        # Mode selector
        mode_frame = tk.LabelFrame(
            self.root, text='Mode',
            bg=self.BG, fg=self.FG,
            font=('Segoe UI', 9)
        )
        mode_frame.grid(row=1, column=0, columnspan=3, sticky='we', padx=12, pady=6)

        self.mode_var = tk.StringVar(value='tigrinya')
        for i, (label, value) in enumerate(self.MODES):
            rb = tk.Radiobutton(
                mode_frame, text=label, variable=self.mode_var, value=value,
                command=self._on_mode_change,
                bg=self.BG, fg=self.FG, selectcolor=self.BTN_BG,
                activebackground=self.BG, activeforeground=self.ACCENT,
                font=('Segoe UI', 10)
            )
            rb.grid(row=0, column=i, padx=10, pady=4)

        # Buttons
        self.start_btn = self._make_button('Start', self._on_start_stop, self.GREEN)
        self.start_btn.grid(row=2, column=0, **pad)

        self.pause_btn = self._make_button('Pause', self._on_pause, self.YELLOW)
        self.pause_btn.grid(row=2, column=1, **pad)

        quit_btn = self._make_button('Quit', self._on_close, self.RED)
        quit_btn.grid(row=2, column=2, **pad)

        for i in range(3):
            self.root.columnconfigure(i, weight=1)

    def _make_button(self, text, command, color):
        return tk.Button(
            self.root, text=text, command=command,
            bg=self.BTN_BG, fg=color, activebackground=self.BTN_ACTIVE,
            activeforeground=color, relief='flat',
            font=('Segoe UI', 10, 'bold'), width=8, pady=4
        )

    # --- Button handlers ---

    def _on_start_stop(self):
        if self.handler is None:
            self.handler = start_keyboard(self.mode_var.get())
            self.paused = False
        else:
            stop_keyboard(self.handler)
            self.handler = None
            self.paused = False
        self._update_ui()

    def _on_pause(self):
        if self.handler is None:
            return
        self.paused = not self.paused
        self.handler.active = not self.paused
        self._update_ui()

    def _on_mode_change(self):
        if self.handler is not None:
            self.handler.switch_mode(InputMode(self.mode_var.get()))
        self._update_ui()

    def _on_close(self):
        if self.handler is not None:
            stop_keyboard(self.handler)
        self.root.destroy()

    # --- UI state sync ---

    def _update_ui(self):
        running = self.handler is not None

        if not running:
            self.status_var.set('Stopped')
            self.status_label.config(fg=self.RED)
            self.start_btn.config(text='Start', fg=self.GREEN)
            self.pause_btn.config(state='disabled')
        elif self.paused:
            self.status_var.set('Paused')
            self.status_label.config(fg=self.YELLOW)
            self.start_btn.config(text='Stop', fg=self.RED)
            self.pause_btn.config(text='Resume', state='normal', fg=self.GREEN)
        else:
            mode_label = next(l for l, v in self.MODES if v == self.mode_var.get())
            self.status_var.set(f'Running  —  {mode_label}')
            self.status_label.config(fg=self.GREEN)
            self.start_btn.config(text='Stop', fg=self.RED)
            self.pause_btn.config(text='Pause', state='normal', fg=self.YELLOW)

## Entry Point

In [ ]:
#| export
def main():
    """Launch the GzKeyboard GUI control panel."""
    root = tk.Tk()
    app = GzKeyboardApp(root)
    root.mainloop()

if __name__ == '__main__':
    main()

## Tests

In [ ]:
#| eval: false
# Smoke test: build the UI and verify initial state (no window displayed)
root = tk.Tk()
root.withdraw()
app = GzKeyboardApp(root)

test_eq(app.handler, None)
test_eq(app.paused, False)
test_eq(app.mode_var.get(), 'tigrinya')
test_eq(app.status_var.get(), 'Stopped')
test_eq(str(app.pause_btn['state']), 'disabled')

root.destroy()
print('GUI smoke test passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()